# Portfolio optimization

Adnan Masood · University of South Florida

Choose what share of the portfolio to place in each of four fictional assets. The goal is to keep modeled return swings small while reaching an estimated 7% return and obeying the position limits. These estimates are assumptions for practice, not a forecast or investment recommendation.

**Dataset:** The four expected returns are assumed annual rates: 6%, 8%, 10%, and 4%. The 4×4 covariance matrix summarizes each asset’s modeled variability and how pairs tend to move together. It is supplied directly; it was not estimated from historical returns in this notebook.

**Task:** Choose four weights that add to 100%, stay between 0% and 50% each, and produce at least a 7% weighted estimated return. Among those choices, minimize variance (modeled return spread); report volatility, its square root, because it is in the same units as return. Check solver success and every limit before interpreting the weights.

**Compare:** Compare the optimized weights with equal weights (25% in each asset) using the same supplied return and covariance estimates. Check both volatility and whether equal weights reach 7%. A lower volatility is not a fair comparison if that portfolio misses the required return or breaks a position limit.

**Runtime:** NumPy stores the estimates and calculates portfolio variance; SciPy’s SLSQP routine searches for weights that reduce variance while meeting the limits. The code checks whether the search succeeded and stops if it did not. Four 20% caps can add to only 80%, so they cannot satisfy the rule that weights total 100%.


In [ ]:
import numpy as np
from scipy.optimize import minimize
# These decimal rates and matrix entries are supplied teaching assumptions, not historical estimates.
mu = np.array([.06,.08,.10,.04])
Sigma = np.array([[.040,.012,.018,.006],[.012,.055,.020,.008],
    [.018,.020,.090,.010],[.006,.008,.010,.025]])
# w @ Sigma @ w is modeled portfolio variance; its square root is volatility in return units.
def variance(w): return w @ Sigma @ w
target_return = .07
asset_cap = .50
constraints = [{"type":"eq","fun":lambda w:w.sum()-1},
    {"type":"ineq","fun":lambda w:w @ mu-target_return}]
# Every share must stay from 0 to 50%; first check that four caps can still fill 100%.
bounds = [(0,asset_cap)]*4
x0 = np.repeat(.25,4)
if len(mu) * asset_cap < 1:
    raise ValueError(f"Infeasible: {len(mu)} asset caps of {100*asset_cap:g}% allow only {100*len(mu)*asset_cap:g}% total allocation; the weights must sum to 100%.")
# SLSQP searches for a low-variance choice while respecting these rules; check its status.
result = minimize(variance,x0,method="SLSQP",bounds=bounds,constraints=constraints)
print("Feasible solve:",result.success,result.message)
if not result.success: raise ValueError("Solver did not find a feasible portfolio")
w = result.x
print("Weights:",w.round(3))
print("Return:",round(w @ mu,4))
print("Volatility:",round(np.sqrt(variance(w)),4))
print("Equal-weight volatility:",round(np.sqrt(variance(x0)),4))
print("Constraint checks:",abs(w.sum()-1)<1e-6,w @ mu >= target_return-1e-6,w.min()>=-1e-6,w.max()<=asset_cap+1e-6)


## Try an experiment

Run the initial example and record its results. Then make one of these changes:

### Raise target return to 9%

Check solver success and every constraint before comparing weights and volatility.

Replace:
```python
target_return = .07
```
With:
```python
target_return = .09
```

### Try infeasible 20% asset caps

Four caps of 20% allow at most 80% total allocation. The diagnostic is the intended result.

Replace:
```python
asset_cap = .50
```
With:
```python
asset_cap = .20
```



In [ ]:
# Raise target_return from 0.07 to 0.09 and inspect solver success first.
# Compare weights and volatility; check whether equal weights meet the new target.


## Break it

Set asset_cap=0.20. Explain why no four-asset allocation can both respect those caps and sum to 100%. Record the diagnostic, then restore a feasible cap before interpreting weights.


In [ ]:
# Compute 4 * 0.20 before running the deliberately infeasible cap.
# After restoring the cap, try Sigma = Sigma + 0.01 * np.eye(4) and compare weights.


## What to submit

- Each asset’s portfolio share, weighted expected return, volatility, solver status, and checks that the shares total 100%, reach the target, and respect every cap.
- An equal-weight comparison under the same estimates, including whether it meets the 7% return requirement.
- Why 20% caps cannot fill a 100% portfolio and a valid covariance sensitivity experiment, with a note that better numerical fit does not make uncertain inputs reliable.

## Optional extension

Add 0.01 to each diagonal entry of the covariance matrix, which raises each asset’s modeled variance while leaving pairwise covariances unchanged. Compare weights, trading needed to move between them, and volatility. Keep the matrix valid; changing one correlation by itself can describe an impossible set of asset movements.
